# YOLO_DRT Docker API — бенчмарк

Контейнер должен быть уже **ready** (`docker compose logs` → «Сервис готов»), потом Run All.

```powershell
cd YOLO_DRT\YOLO_DOCKER
docker compose up -d
pip install -r notebooks/requirements.txt
jupyter notebook notebooks/docker_api_benchmark.ipynb
```

In [ ]:
from pathlib import Path

API_BASE = "http://127.0.0.1:8080"
VIDEO_PATH = Path(r"C:/Users/Shtefan/Pictures/Camera Roll/WIN_20260625_15_23_42_Pro.mp4")
PROMPT = "person"
MAX_DURATION_SECONDS = None   # None = всё видео
REPEAT_RUNS = 0

RESULTS_DIR = Path("benchmark_results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
import json
import time
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Callable

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 110


class ApiError(RuntimeError):
    pass


@dataclass
class Tick:
    t_wall: float
    status: str
    current: int
    total: int
    percent: float
    fps: float
    elapsed_sec: float
    eta_seconds: float
    gpu_mem_mb: float
    gpu_util_pct: float
    instances_peak: int


@dataclass
class RunResult:
    job_id: str
    wall_sec: float
    status: str
    job: dict[str, Any]
    ticks: list[Tick] = field(default_factory=list)
    files: dict[str, bytes] = field(default_factory=dict)
    error: str | None = None

    @property
    def result(self) -> dict:
        return self.job.get("result") or {}

    @property
    def record(self) -> dict:
        return self.result.get("record") or {}


class DockerApi:
    """Клиент под yolo-drt-api FastAPI (api/main.py)."""

    def __init__(self, base: str) -> None:
        self.base = base.rstrip("/")
        self.s = requests.Session()

    def health(self) -> dict:
        r = self.s.get(f"{self.base}/health", timeout=30)
        r.raise_for_status()
        return r.json()

    def upload_job(self, video: Path, *, prompt: str, max_sec: float | None) -> str:
        data = {"prompt": prompt}
        if max_sec is not None:
            data["max_duration_seconds"] = str(max_sec)
        with video.open("rb") as f:
            r = self.s.post(
                f"{self.base}/v1/jobs/upload",
                data=data,
                files={"file": (video.name, f, "application/octet-stream")},
                timeout=3600,
            )
        if r.status_code >= 400:
            raise ApiError(f"upload {r.status_code}: {r.text}")
        return r.json()["job_id"]

    def job(self, job_id: str) -> dict:
        r = self.s.get(f"{self.base}/v1/jobs/{job_id}", timeout=30)
        r.raise_for_status()
        return r.json()

    def artifacts(self, job_id: str) -> list[dict]:
        r = self.s.get(f"{self.base}/v1/jobs/{job_id}/artifacts", timeout=60)
        r.raise_for_status()
        return r.json().get("files") or []

    def download(self, job_id: str, name: str) -> bytes:
        r = self.s.get(f"{self.base}/v1/jobs/{job_id}/artifacts/{name}", timeout=300)
        r.raise_for_status()
        return r.content

    def runs(self) -> list[dict]:
        r = self.s.get(f"{self.base}/v1/runs", timeout=30)
        r.raise_for_status()
        return r.json().get("runs") or []

    def poll_until_done(
        self,
        job_id: str,
        *,
        interval: float = 0.5,
        timeout: float = 7200,
        on_tick: Callable[[Tick], None] | None = None,
    ) -> tuple[dict, list[Tick]]:
        ticks: list[Tick] = []
        t0 = time.time()
        while time.time() - t0 < timeout:
            j = self.job(job_id)
            p = j.get("progress") or {}
            tick = Tick(
                t_wall=time.time() - t0,
                status=j.get("status", ""),
                current=int(p.get("current", 0)),
                total=int(p.get("total", 0)),
                percent=float(p.get("percent", 0)),
                fps=float(p.get("fps", 0)),
                elapsed_sec=float(p.get("elapsed_sec", 0)),
                eta_seconds=float(p.get("eta_seconds", 0)),
                gpu_mem_mb=float(p.get("gpu_mem_mb", 0)),
                gpu_util_pct=float(p.get("gpu_util_pct", 0)),
                instances_peak=int(p.get("instances_peak", 0)),
            )
            ticks.append(tick)
            if on_tick:
                on_tick(tick)
            if tick.status in ("done", "error", "cancelled"):
                return j, ticks
            time.sleep(interval)
        raise ApiError(f"Job {job_id} timeout {timeout}s")


api = DockerApi(API_BASE)

## 1. Проверка Docker API

In [ ]:
health = api.health()
print("status:", health["status"])
print("engines:", health.get("engines_ready"))
if health["status"] != "ready":
    raise RuntimeError(f"API не ready — дождись загрузки контейнера. {health.get('message', '')}")

## 2. Запуск inference job

In [ ]:
def run_api_benchmark(*, label: str = "run", fetch_artifacts: bool = True) -> RunResult:
    if not VIDEO_PATH.is_file():
        raise FileNotFoundError(VIDEO_PATH)

    print(f"=== {label} === upload {VIDEO_PATH.name}")
    job_id = api.upload_job(VIDEO_PATH, prompt=PROMPT, max_sec=MAX_DURATION_SECONDS)
    print("job_id:", job_id)

    t0 = time.time()
    last_sec = -1

    def on_tick(t: Tick) -> None:
        nonlocal last_sec
        sec = int(t.t_wall)
        if sec != last_sec and t.total > 0:
            last_sec = sec
            print(f"  [{sec:4d}s] {t.percent:5.1f}%  {t.current}/{t.total}  fps={t.fps:.1f}  gpu={t.gpu_util_pct:.0f}%")

    job, ticks = api.poll_until_done(job_id, interval=0.5, timeout=7200, on_tick=on_tick)
    wall = time.time() - t0
    status = job.get("status", "")
    err = (job.get("result") or {}).get("error")

    out = RunResult(job_id=job_id, wall_sec=wall, status=status, job=job, ticks=ticks, error=err)

    if status == "error":
        print(f"FAILED: {err}")
        return out

    res = out.result
    rec = out.record
    print(f"\nDONE  wall={wall:.1f}s  pipeline={res.get('elapsed_sec', 0):.1f}s  fps={res.get('fps_processed', 0):.1f}")
    print(f"  run_id: {res.get('run_id')}")
    print(f"  out_dir: {res.get('out_dir')}")
    if rec:
        print(f"  video: {rec.get('resolution')} @ {rec.get('video_fps')} fps, frames={rec.get('frames')}/{rec.get('source_frames')}")
        pipe = rec.get("pipeline") or {}
        if pipe.get("smart_ram_budget_gb"):
            print(f"  Smart RAM: budget={pipe['smart_ram_budget_gb']}GB peak≈{pipe.get('smart_ram_peak_gb')}GB")
        gs = rec.get("gpu_stats") or {}
        if gs:
            print(f"  GPU util avg/peak: {gs.get('avg_gpu_util_pct')}/{gs.get('peak_gpu_util_pct')}%  VRAM peak: {gs.get('peak_mem_used_mb')} MB")

    if fetch_artifacts and status == "done":
        for item in api.artifacts(job_id):
            name = item["name"]
            if name.endswith((".json", ".jsonl", ".txt", ".png")):
                out.files[name] = api.download(job_id, name)
                print(f"  ↓ {name} ({len(out.files[name])} B)")

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = RESULTS_DIR / f"{stamp}_{label}.json"
    save_path.write_text(
        json.dumps(
            {"label": label, "job": job, "ticks": [t.__dict__ for t in ticks], "wall_sec": wall},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"  saved {save_path.name}")
    return out


RUN = run_api_benchmark(label="main")
if RUN.status != "done":
    raise ApiError(RUN.error or RUN.status)

## 3. Графики — progress polling (GET /v1/jobs/{id})

In [ ]:
df = pd.DataFrame([t.__dict__ for t in RUN.ticks])
if df.empty:
    raise ApiError("Нет samples — job не отработал?")

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(f"Job {RUN.job_id[:12]}…  wall={RUN.wall_sec:.1f}s", fontsize=13)

axes[0].plot(df.t_wall, df.percent, color="#2563eb", lw=2)
axes[0].fill_between(df.t_wall, df.percent, alpha=0.12, color="#2563eb")
axes[0].set_ylabel("%")
axes[0].set_ylim(0, 105)
axes[0].set_title("Progress")

axes[1].plot(df.t_wall, df.fps.replace(0, pd.NA), color="#059669", lw=1.5)
med = df.fps.replace(0, pd.NA).dropna().median()
if med == med:
    axes[1].axhline(med, ls="--", c="#64748b", label=f"median {med:.1f}")
    axes[1].legend()
axes[1].set_ylabel("FPS")
axes[1].set_title("Inference FPS (from API progress)")

ax = axes[2]
ax.plot(df.t_wall, df.gpu_util_pct, color="#dc2626", label="GPU util %")
ax2 = ax.twinx()
ax2.plot(df.t_wall, df.gpu_mem_mb, color="#7c3aed", alpha=0.85, label="VRAM MB")
ax.set_ylabel("GPU util %")
ax2.set_ylabel("VRAM MB")
ax.set_xlabel("Wall time (s)")
ax.set_title("GPU during job (API progress)")
plt.tight_layout()
plt.show()

## 4. GPU samples из контейнера (`*_gpu_samples.json`)

In [ ]:
gpu_name = next((n for n in RUN.files if n.endswith("_gpu_samples.json")), None)
if not gpu_name:
    print("Нет gpu_samples — возможно job упал до записи или NVML недоступен")
else:
    gdf = pd.DataFrame(json.loads(RUN.files[gpu_name].decode()))
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(gdf.t_sec, gdf.gpu_util_pct, color="#dc2626")
    axes[0].set_ylabel("GPU util %")
    axes[0].set_title("NVML inside container (0.5s interval)")
    axes[1].plot(gdf.t_sec, gdf.mem_used_mb, color="#7c3aed")
    axes[1].set_ylabel("VRAM MB")
    axes[1].set_xlabel("Inference time (s)")
    plt.tight_layout()
    plt.show()
    display(gdf[["gpu_util_pct", "mem_used_mb"]].describe().round(1))

## 5. Realtime ratio + Smart RAM + pipeline

In [ ]:
rec = RUN.record
res = RUN.result
pipe = rec.get("pipeline") or {}
gs = rec.get("gpu_stats") or {}
stats = rec.get("stats_summary") or {}

src = int(rec.get("source_frames") or res.get("frames") or 0)
vfps = float(rec.get("video_fps") or 30)
video_sec = src / vfps if vfps else 0
infer_sec = float(res.get("elapsed_sec") or RUN.wall_sec)
rt = infer_sec / video_sec if video_sec else float("nan")

summary = pd.DataFrame(
    {
        "metric": [
            "run_id", "wall_sec (API)", "pipeline_sec", "fps_processed",
            "source_frames", "processed_frames", "frame_stride",
            "video_duration_sec", "realtime_ratio", "realtime_ok (<1.15×)",
            "frame_source", "job_batch", "windowed_decode",
            "smart_ram_budget_gb", "smart_ram_peak_gb", "smart_ram_window_gb",
            "gpu_util_avg", "gpu_util_peak", "vram_peak_mb",
        ],
        "value": [
            res.get("run_id"), round(RUN.wall_sec, 2), round(infer_sec, 2),
            round(float(res.get("fps_processed") or 0), 2),
            src, rec.get("frames"), stats.get("frame_stride") or pipe.get("frame_stride"),
            round(video_sec, 2), round(rt, 3), rt <= 1.15 if video_sec else None,
            pipe.get("frame_source_mode"), pipe.get("effective_batch_size"), pipe.get("windowed_decode"),
            pipe.get("smart_ram_budget_gb"), pipe.get("smart_ram_peak_gb"), pipe.get("smart_ram_window_gb"),
            gs.get("avg_gpu_util_pct"), gs.get("peak_gpu_util_pct"), gs.get("peak_mem_used_mb"),
        ],
    }
)
display(summary)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    ["Video", "Pipeline", "Wall+upload"],
    [video_sec, infer_sec, RUN.wall_sec],
    color=["#f59e0b", "#059669", "#6366f1"],
)
ax.set_ylabel("seconds")
ax.set_title(f"Realtime {rt:.2f}×  ({'OK ~1×' if rt <= 1.15 else 'медленнее видео'})")
for b, v in zip(bars, [video_sec, infer_sec, RUN.wall_sec]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.2, f"{v:.1f}s", ha="center")
plt.tight_layout()
plt.show()

## 6. Run summary + модели (из artifact или record)

In [ ]:
summary_name = next((n for n in RUN.files if n.endswith("_run_summary.json")), None)
if summary_name:
    full = json.loads(RUN.files[summary_name].decode())
    print("Models:", full.get("models"))
    print("Charts:", full.get("charts"))
    mem = full.get("memory") or {}
    if mem:
        print(f"Frame RAM: {mem.get('frame_human')}  batch: {mem.get('batch_human')}  preloaded={mem.get('preloaded')}")
else:
    print("Models:", rec.get("models"))

chart_png = next((n for n in RUN.files if n.endswith("_chart_gpu.png")), None)
if chart_png:
    from IPython.display import Image
    display(Image(data=RUN.files[chart_png]))

## 7. Повторные прогоны (опционально)

Поставь `REPEAT_RUNS = 3` в config и перезапусти ячейку ниже.

In [ ]:
ALL: list[RunResult] = [RUN]
for i in range(REPEAT_RUNS):
    ALL.append(run_api_benchmark(label=f"repeat_{i+1}", fetch_artifacts=False))

rows = []
for i, r in enumerate(ALL):
    if r.status != "done":
        continue
    rec = r.record
    res = r.result
    src = int(rec.get("source_frames") or 0)
    vfps = float(rec.get("video_fps") or 30)
    vsec = src / vfps if vfps else 0
    isec = float(res.get("elapsed_sec") or r.wall_sec)
    rows.append({
        "#": i, "run_id": res.get("run_id"), "wall_s": r.wall_sec,
        "fps": res.get("fps_processed"), "realtime×": round(isec / vsec, 3) if vsec else None,
    })

cmp = pd.DataFrame(rows)
if len(cmp) > 1:
    display(cmp)
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].bar(cmp["#"], cmp["fps"], color="#059669")
    ax[0].set_title("FPS processed")
    ax[1].bar(cmp["#"], cmp["realtime×"], color="#2563eb")
    ax[1].axhline(1.0, ls="--", c="red", alpha=0.5)
    ax[1].set_title("Realtime ratio")
    ax[2].bar(cmp["#"], cmp["wall_s"], color="#6366f1")
    ax[2].set_title("Wall sec")
    plt.tight_layout()
    plt.show()
elif REPEAT_RUNS == 0:
    print("REPEAT_RUNS=0 — один прогон. Увеличь в config для сравнения.")

## 8. История — GET /v1/runs

In [ ]:
hist = pd.DataFrame(api.runs())
print(f"Runs in container index: {len(hist)}")
if not hist.empty:
    cols = [c for c in ["run_id", "elapsed_sec", "fps_processed", "frames", "source_frames", "resolution"] if c in hist.columns]
    display(hist[cols].head(15) if cols else hist.head(15))
    if "fps_processed" in hist.columns and len(hist) > 1:
        tail = hist.head(25).iloc[::-1].reset_index(drop=True)
        plt.figure(figsize=(12, 4))
        plt.plot(tail.index, tail["fps_processed"], "o-", color="#059669")
        plt.ylabel("fps_processed")
        plt.xlabel("last runs (newest first)")
        plt.title("/v1/runs history (container /data/output)")
        plt.tight_layout()
        plt.show()